# Task 1: Pattern Mining Module

This notebook implements the frequent pattern mining task. It includes:
1. Loading the training data.
2. Preparing data into a transactional format.
3. Mining frequent itemsets using FP-Growth (`mlxtend`).
4. Generating association rules with standard metrics (support, confidence, lift).
5. A placeholder for implementing user-specific scoring logic.

## Table Of Contents:
1. [Import Libraries](#first-bullet)
2. [Second Bullet Header](#second-bullet)

## __1. Import Libraries <a class="anchor" id="first-bullet"></a>__

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
from datetime import datetime
import numpy as np 

### 2. Data Loading Function <a class="anchor" id="first-bullet"></a>

In [2]:
import pandas as pd # Import the pandas library for data manipulation

def load_data(filepath):
    """
    Loads transaction data from a specified CSV file, performs basic preprocessing,
    and returns a pandas DataFrame.

    Args:
        filepath (str): The path to the CSV file to be loaded.

    Returns:
        pandas.DataFrame or None: A DataFrame containing the loaded and preprocessed
                                  data, or None if an error occurs during loading
                                  (e.g., file not found, missing required columns).
    """
    try:
        # Attempt to read the CSV file into a pandas DataFrame.
        df = pd.read_csv(filepath)

        # --- Basic Preprocessing ---
        # This section handles initial data cleaning and type conversions.

        # Check if a 'Date' column exists in the DataFrame.
        if 'Date' in df.columns:
            # If it exists, convert the 'Date' column to datetime objects.
            # This is crucial for time-based analysis.
            df['Date'] = pd.to_datetime(df['Date'])
        else:
            # If 'Date' column doesn't exist, try to create it from 'year', 'month', 'day' columns.
            try:
                # Combine 'year', 'month', and 'day' columns to create a datetime 'Date' column.
                df['Date'] = pd.to_datetime(df[['year', 'month', 'day']])
            except KeyError:
                # If 'year', 'month', or 'day' columns are missing, creating 'Date' fails.
                # Print a warning message as time-related features might not work correctly.
                print("Warning: Cannot find/create 'Date' column. Timing features might fail.")

        # Check if a 'user_id' column exists.
        if 'user_id' in df.columns:
            # If it exists, convert the 'user_id' column to the string data type.
            # Ensures consistent handling of user identifiers.
            df['user_id'] = df['user_id'].astype(str)

        # Check if an 'itemDescription' column exists.
        if 'itemDescription' in df.columns:
            # If it exists, convert the column to string type.
            # Then, remove any leading or trailing whitespace from the descriptions.
            df['itemDescription'] = df['itemDescription'].astype(str).str.strip()
        else:
            # If the 'itemDescription' column is missing, raise a ValueError.
            # This column is considered essential for the intended analysis (e.g., association rule mining).
            raise ValueError("Missing 'itemDescription' column needed for mining.")

        # If loading and preprocessing were successful, print a confirmation message.
        # Include the source filepath and the dimensions (shape) of the loaded DataFrame.
        print(f"Data loaded successfully from {filepath}. Shape: {df.shape}")
        # Return the processed DataFrame.
        return df

    # Handle the specific error when the file cannot be found at the given filepath.
    except FileNotFoundError:
        # Print an informative error message.
        print(f"Error: File not found at {filepath}")
        # Return None to indicate that data loading failed.
        return None

    # Handle any other unexpected exceptions that might occur during the process.
    except Exception as e:
        # Print a general error message, including the specific exception details.
        print(f"An error occurred during data loading: {e}")
        # Return None to indicate that data loading failed due to an unexpected error.
        return None

## 3. Data Preparation Function

In [3]:
def prepare_data_for_mining(df, user_col='user_id', item_col='itemDescription', date_col='Date'):
    """
    Transforms the dataframe into a list of transactions suitable for mlxtend.
    Groups items purchased by the same user on the same date as one transaction.
    """
    # Check if essential columns exist, handle potential absence of date_col if previously warned
    required_cols = [user_col, item_col]
    if date_col in df.columns:
        required_cols.append(date_col)
    else: # Group only by user if date is missing
        print(f"Warning: Grouping transactions without '{date_col}'. This might merge distinct shopping trips.")
        grouping_cols = [user_col]
        
    if df is None or not all(col in df.columns for col in required_cols):
         print("Error: Missing required columns for data preparation.")
         return None, None
    
    grouping_cols = [user_col, date_col] if date_col in df.columns else [user_col]

    print("Preparing data for pattern mining...")
    # Group by user and date (if available) to create transaction baskets
    transactions_df = df.groupby(grouping_cols)[item_col].apply(list).reset_index()
    transaction_list = transactions_df[item_col].tolist()

    # Use TransactionEncoder to transform data into a one-hot encoded format
    te = TransactionEncoder()
    # Handle potential empty lists if grouping results in them
    transaction_list_filtered = [t for t in transaction_list if t] 
    if not transaction_list_filtered:
        print("Warning: No valid transactions found after grouping.")
        return pd.DataFrame(), df # Return empty DataFrame but original df
        
    te_ary = te.fit(transaction_list_filtered).transform(transaction_list_filtered)
    one_hot_df = pd.DataFrame(te_ary, columns=te.columns_)
    print(f"Data prepared into {one_hot_df.shape[0]} transactions.")
    return one_hot_df, df # Return original df too, needed for user-specific scoring

## 4. Pattern Mining and Scoring Function (Module)

In [4]:
def mine_and_score_patterns(
    train_filepath,
    min_support_threshold=0.01,
    metric="lift", # Metric for filtering association rules (e.g., lift, confidence)
    min_metric_threshold=1.0, # Minimum value for the chosen metric
    user_id=None # Optional user_id for user-specific scoring
    ):
    """
    Mines frequent patterns and scores them globally and optionally for a user.

    Args:
        train_filepath (str): Path to the training data CSV.
        min_support_threshold (float): Minimum support for frequent itemsets.
        metric (str): Metric to filter association rules ('lift', 'confidence', etc.).
        min_metric_threshold (float): Minimum threshold for the chosen metric.
        user_id (str, optional): The user ID for user-specific scoring. Defaults to None.

    Returns:
        pandas.DataFrame: DataFrame containing association rules with standard metrics
                          and potentially a user-specific score column.
                          Returns None if an error occurs.
    """
    # --- Load and Prepare ---
    raw_train_df = load_data(train_filepath)
    if raw_train_df is None:
        return None
    one_hot_df, original_df = prepare_data_for_mining(raw_train_df)
    if one_hot_df is None:
        return None
    # Handle case where preparation resulted in no transactions
    if one_hot_df.empty:
        print("Skipping mining as no transactions were prepared.")
        return pd.DataFrame()

    # --- Mine Frequent Itemsets using FP-Growth ---
    print(f"Mining frequent itemsets with min_support={min_support_threshold}...")
    frequent_itemsets = fpgrowth(one_hot_df, min_support=min_support_threshold, use_colnames=True)
    print(f"Found {len(frequent_itemsets)} frequent itemsets.")

    if frequent_itemsets.empty:
        print("No frequent itemsets found with the given support threshold.")
        return pd.DataFrame() # Return empty DataFrame

    # --- Generate Association Rules ---
    print(f"Generating association rules with metric='{metric}' >= {min_metric_threshold}...")
    # Ensure frequent_itemsets is not empty before proceeding
    if frequent_itemsets.empty:
        print("Cannot generate rules from empty frequent itemsets.")
        return pd.DataFrame()
        
    rules = association_rules(frequent_itemsets, metric=metric, min_threshold=min_metric_threshold)
    print(f"Generated {len(rules)} rules meeting the threshold.")

    if rules.empty:
        print("No rules generated meeting the metric threshold.")
        return rules # Return empty rules DataFrame

    # --- Add User-Specific Score (Placeholder - NEEDS IMPLEMENTATION) ---
    rules['user_specific_score'] = np.nan # Initialize column with NaN

    if user_id:
        print(f"Calculating user-specific scores for user_id: {user_id}...")
        # Filter original data for the specific user
        user_history = original_df[original_df['user_id'] == user_id]

        if user_history.empty:
            print(f"Warning: No history found for user_id {user_id}. Cannot calculate user-specific scores.")
        else:
            # !!! THIS IS WHERE YOUR CUSTOM LOGIC GOES !!!
            # Iterate through the 'rules' DataFrame and calculate your chosen/proposed metric for each rule
            # based on 'user_history'.
            # Example ideas:
            # - User-specific confidence: How often did this user buy 'consequents' when they bought 'antecedents'?
            # - Recency weighting: Give higher scores to rules involving items this user bought recently.
            # - Combined metric: e.g., global_lift * user_confidence * recency_weight

            # Example placeholder calculation (replace with your actual logic):
            user_items = set(user_history['itemDescription'])
            for index, rule in rules.iterrows():
                # --- Replace this logic ---
                # Simple example: Score based on how many items from the rule the user bought
                rule_items = set(rule['antecedents']).union(set(rule['consequents']))
                common_items = rule_items.intersection(user_items)
                # This is a very basic score, likely needs improvement based on your research/metric definition
                score = len(common_items) / len(rule_items) if rule_items else 0
                rules.loc[index, 'user_specific_score'] = score
                # --- End replace ---

            print("User-specific scores calculated (using placeholder logic).")

    else:
        print("No user_id provided, skipping user-specific scoring.")


    # --- Sort and Return ---
    # Sort rules for better readability (e.g., by lift or confidence)
    rules = rules.sort_values(by=metric, ascending=False).reset_index(drop=True)

    return rules

## 5. Example Usage

Now, let's run the functions defined above.

In [5]:
# --- Configuration ---

train_data_filepath = 'Groceries data train.csv'
min_support = 0.005 # Adjust based on your dataset size and item frequency
min_lift = 1.1      # Adjust based on desired rule strength

### 5a. Mining Global Patterns

In [6]:
print("\n--- Mining Global Patterns ---")
global_rules = mine_and_score_patterns(
    train_filepath=train_data_filepath,
    min_support_threshold=min_support,
    metric="lift",
    min_metric_threshold=min_lift
)

if global_rules is not None and not global_rules.empty:
    print("\nTop 10 Global Rules (Sorted by Lift):")
    # Displaying the rules - adjust display options if needed
    with pd.option_context('display.max_rows', 10, 'display.max_columns', None, 'display.width', 1000):
        display(global_rules.head(10)) 
else:
    print("No global rules generated or an error occurred.")


--- Mining Global Patterns ---
An error occurred during data loading: time data "13/01/2014" doesn't match format "%m/%d/%Y", at position 12. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
No global rules generated or an error occurred.


### 5b. Mining with User-Specific Scoring

In [7]:
print("\n" + "="*50 + "\n")
print("--- Mining with User-Specific Scoring (Example User) ---")
# !!! REPLACE 'some_user_id' WITH AN ACTUAL ID FROM YOUR TRAINING DATA !!!
example_user = 'some_user_id' 

user_scored_rules = mine_and_score_patterns(
    train_filepath=train_data_filepath,
    min_support_threshold=min_support,
    metric="lift",
    min_metric_threshold=min_lift,
    user_id=example_user
)

if user_scored_rules is not None and not user_scored_rules.empty:
    # Filter out rules where user score couldn't be calculated (NaNs)
    user_scored_rules_filtered = user_scored_rules.dropna(subset=['user_specific_score'])
    
    if not user_scored_rules_filtered.empty:
        # Sort by the user-specific score
        user_scored_rules_sorted = user_scored_rules_filtered.sort_values(by='user_specific_score', ascending=False)

        print(f"\nTop 10 Rules for User '{example_user}' (Sorted by User Score - Placeholder Logic):")
        # Displaying the rules
        with pd.option_context('display.max_rows', 10, 'display.max_columns', None, 'display.width', 1000):
            display(user_scored_rules_sorted.head(10))
    else:
        print(f"No rules could be scored for user '{example_user}' (or placeholder logic resulted in NaNs).")
else:
    print(f"No rules generated or an error occurred when processing for user '{example_user}'.")



--- Mining with User-Specific Scoring (Example User) ---
An error occurred during data loading: time data "13/01/2014" doesn't match format "%m/%d/%Y", at position 12. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
No rules generated or an error occurred when processing for user 'some_user_id'.


## Next Steps

1.  **Replace Placeholders:** Update the `train_data_filepath` and `example_user` variables with your actual data.
2.  **Tune Thresholds:** Adjust `min_support` and `min_lift` (or other metric threshold) based on the results to get a meaningful number of rules.
3.  **Implement User-Specific Scoring:** The most important step is to replace the placeholder logic in the `mine_and_score_patterns` function with your chosen or proposed user-specific scoring metric. This requires research and careful consideration based on the project requirements.